In [2]:
import pandas as pd
import numpy as np
import re

secop_df = pd.read_parquet('secop_bienes.parquet')
print('Filas:', secop_df.shape[0])
print('Columnas:', secop_df.shape[1])

Filas: 196391
Columnas: 36


In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
secop_df.columns

Index(['id_contrato', 'nombre_entidad', 'departamento', 'ciudad', 'orden',
       'sector', 'rama', 'entidad_centralizada', 'tipo_de_contrato',
       'modalidad_de_contratacion', 'codigo_de_categoria_principal',
       'objeto_del_contrato', 'fecha_de_firma', 'fecha_de_inicio_del_contrato',
       'fecha_de_fin_del_contrato', 'duraci_n_del_contrato',
       'condiciones_de_entrega', 'proveedor_adjudicado', 'documento_proveedor',
       'es_pyme', 'es_grupo', 'g_nero_representante_legal',
       'nacionalidad_representante_legal', 'valor_del_contrato',
       'valor_pagado', 'valor_facturado', 'valor_pendiente_de_pago',
       'dias_adicionados', 'habilita_pago_adelantado',
       'el_contrato_puede_ser_prorrogado', 'origen_de_los_recursos',
       'destino_gasto', 'estado_contrato', 'liquidaci_n',
       'obligaci_n_ambiental', 'urlproceso'],
      dtype='str')

In [4]:
secop_df.dtypes

id_contrato                                    str
nombre_entidad                                 str
departamento                                   str
ciudad                                         str
orden                                          str
sector                                         str
rama                                           str
entidad_centralizada                           str
tipo_de_contrato                               str
modalidad_de_contratacion                      str
codigo_de_categoria_principal                  str
objeto_del_contrato                            str
fecha_de_firma                      datetime64[us]
fecha_de_inicio_del_contrato        datetime64[us]
fecha_de_fin_del_contrato           datetime64[us]
duraci_n_del_contrato                          str
condiciones_de_entrega                         str
proveedor_adjudicado                           str
documento_proveedor                            str
es_pyme                        

In [5]:
def fragmento_sospechoso(variable_temporal_columna):
    fragmentos = variable_temporal_columna.split('_')
    return any(len(frag) == 1 for frag in fragmentos)

columnas_sospechosas = [col for col in secop_df.columns if fragmento_sospechoso(col)]

print("Columnas con posible problema de encoding:")
for col in columnas_sospechosas:
    print(col)

Columnas con posible problema de encoding:
duraci_n_del_contrato
g_nero_representante_legal
liquidaci_n
obligaci_n_ambiental


In [6]:
correccion_columnas = {
    'duraci_n_del_contrato': 'duracion_del_contrato',
    'g_nero_representante_legal': 'genero_representante_legal',
    'liquidaci_n': 'liquidacion',
    'obligaci_n_ambiental': 'obligacion_ambiental',
}

secop_df = secop_df.rename(columns=correccion_columnas)
print("Columnas corregidas:")
for original, corregida in correccion_columnas.items():
    print(f"  {original} -> {corregida}")

Columnas corregidas:
  duraci_n_del_contrato -> duracion_del_contrato
  g_nero_representante_legal -> genero_representante_legal
  liquidaci_n -> liquidacion
  obligaci_n_ambiental -> obligacion_ambiental


## Validación sistemática de calidad de datos

Antes de fijar el Top 5 de atributos, revisamos de forma sistemática (no solo mirando `describe()`) los problemas de calidad típicos: nulos, fechas fuera de rango o inconsistentes entre sí, valores negativos donde no deberían existir, y categorías "centinela" (`No Definido`/`No aplica`) que en realidad son valores faltantes disfrazados de texto.

In [ ]:
print("Valores nulos por columna (solo columnas con al menos uno):")
nulos = secop_df.isna().sum()
print(nulos[nulos > 0])

rango_min, rango_max = '2016-01-01', '2025-12-31'
cols_fecha = ['fecha_de_firma', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato']

print("\nFechas fuera del rango esperado (2016-2025):")
for col in cols_fecha:
    fuera_rango = ((secop_df[col] < rango_min) | (secop_df[col] > rango_max)).sum()
    print(f"  {col}: {fuera_rango} filas fuera de rango (min={secop_df[col].min()}, max={secop_df[col].max()})")

fin_antes_inicio = (secop_df['fecha_de_fin_del_contrato'] < secop_df['fecha_de_inicio_del_contrato']).sum()
print(f"\nFilas con fecha_de_fin_del_contrato anterior a fecha_de_inicio_del_contrato: {fin_antes_inicio}")

print("\nValores negativos en columnas monetarias:")
cols_valor = ['valor_del_contrato', 'valor_pagado', 'valor_facturado', 'valor_pendiente_de_pago']
for col in cols_valor:
    negativos = (secop_df[col] < 0).sum()
    print(f"  {col}: {negativos} filas negativas")

print("\nCategorías centinela ('No Definido'/'No aplica') por columna de texto:")
columnas_texto = secop_df.select_dtypes(exclude=['number', 'datetime']).columns
centinelas = {'no definido', 'no aplica'}
for col in columnas_texto:
    conteo = secop_df[col].str.strip().str.lower().isin(centinelas).sum()
    if conteo > 0:
        print(f"  {col}: {conteo}")

### Tratamiento aplicado

Con base en los hallazgos anteriores, se decidió lo siguiente (sin eliminar filas, para no perder información válida de otras columnas):

1. **Categorías centinela ("No Definido"/"No aplica")** → se estandarizan a `NA` explícito en todas las columnas de texto. Se trataban como una categoría más en los `value_counts`, pero son valores faltantes disfrazados; el caso más notorio es `condiciones_de_entrega` con 25.4% de filas afectadas.
2. **Fechas fuera de rango o inconsistentes** (`fecha_de_inicio`/`fecha_de_fin` fuera de 2016–2025, o `fecha_de_fin < fecha_de_inicio`) → no se eliminan ni se imputan; se marcan con una columna booleana `fecha_sospechosa` para poder excluirlas puntualmente de análisis de duración/plazo más adelante, sin perder el resto de la fila.
3. **`valor_pendiente_de_pago` negativo (144 filas)** → se investigó el patrón: el 100% de esos casos tiene `valor_pagado > valor_del_contrato`, es decir, sobrepago real y no un error de captura. Se deja el valor tal cual; es una señal de posible interés para supervisión, no un dato a corregir.

In [ ]:
centinelas_texto = ['no definido', 'no aplica']
columnas_texto = secop_df.select_dtypes(exclude=['number', 'datetime']).columns

for col in columnas_texto:
    es_centinela = secop_df[col].str.strip().str.lower().isin(centinelas_texto)
    secop_df.loc[es_centinela, col] = pd.NA

print("Categorías centinela convertidas a NA. Nulos por columna (top 10):")
nulos_actualizados = secop_df.isna().sum()
print(nulos_actualizados[nulos_actualizados > 0].sort_values(ascending=False).head(10))

In [ ]:
rango_min, rango_max = '2016-01-01', '2025-12-31'

secop_df['fecha_sospechosa'] = (
    (secop_df['fecha_de_inicio_del_contrato'] < rango_min) |
    (secop_df['fecha_de_inicio_del_contrato'] > rango_max) |
    (secop_df['fecha_de_fin_del_contrato'] < rango_min) |
    (secop_df['fecha_de_fin_del_contrato'] > rango_max) |
    (secop_df['fecha_de_fin_del_contrato'] < secop_df['fecha_de_inicio_del_contrato'])
)

print("Contratos marcados con fecha_sospechosa:", secop_df['fecha_sospechosa'].sum(),
      f"({secop_df['fecha_sospechosa'].mean():.1%} del total)")

## Top 5 de atributos clave

El taller señala explícitamente que el **valor**, la **modalidad de contratación**, el **sector**, el **tipo de entidad** y el **destino del gasto** son las características que se espera estén asociadas con desviaciones en la ejecución de un contrato (adiciones de plazo, presupuesto no ejecutado, cierre sin liquidar). Por eso estos son los 5 atributos priorizados para el análisis univariado inicial:

1. **`valor_del_contrato`** — tamaño económico del contrato; a mayor valor, mayor impacto potencial de una desviación.
2. **`sector`** — área de gobierno a la que pertenece la entidad contratante.
3. **`modalidad_de_contratacion`** — mecanismo usado para adjudicar el contrato (licitación, mínima cuantía, etc.), asociado en la literatura de contratación pública a distintos niveles de riesgo y control.
4. **`orden`** — tipo de entidad (Nacional vs. Territorial), relevante porque la capacidad de supervisión varía según el nivel de gobierno.
5. **`destino_gasto`** — si el gasto es de funcionamiento o de inversión, lo cual afecta el tipo de seguimiento presupuestal esperado.

In [7]:
secop_df.head()

,id_contrato,nombre_entidad,departamento,ciudad,orden,sector,rama,entidad_centralizada,tipo_de_contrato,modalidad_de_contratacion,codigo_de_categoria_principal,objeto_del_contrato,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,duracion_del_contrato,condiciones_de_entrega,proveedor_adjudicado,documento_proveedor,es_pyme,es_grupo,genero_representante_legal,nacionalidad_representante_legal,valor_del_contrato,valor_pagado,valor_facturado,valor_pendiente_de_pago,dias_adicionados,habilita_pago_adelantado,el_contrato_puede_ser_prorrogado,origen_de_los_recursos,destino_gasto,estado_contrato,liquidacion,obligacion_ambiental,urlproceso
0,CO1.PCCNTR.1000001,PARQUES NACIONALES NATURALES DE COLOMBIA - DIRECCION TERRITORIAL AMAZONIA,Distrito Capital de Bogotá,No Definido,Territorial,Ambiente y Desarrollo Sostenible,Ejecutivo,Centralizada,Suministros,Mínima cuantía,V1.50161509,Contratar a monto agotable por el sistema de precios unitarios; sin formula de reajuste el suministro de elementos de aseo y limpieza e insumos de cafetería y restaurante; para los Parques Nacionales Naturales de Amacayacu; Cahuinarí; Río Puré y Yaigojé Apaporis,2019-06-19,2019-06-18,2019-12-31,No definido,Como acordado previamente,INDUSTRIAS GUERRERO Y COMPAÑIA S.A.S.,830130048,Si,No,No Definido,CO,17897916.0,17896237,17896237,1679,0,No Definido,Si,Distribuido,Funcionamiento,Cerrado,Si,No,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.846428&isFromPublicArea=True&isModal=true&asPopupView=true'}
1,CO1.PCCNTR.1000507,AGENCIA LOGISTICA DE LAS FUERZAS MILITARES,Distrito Capital de Bogotá,Bogotá,Nacional,defensa,Ejecutivo,Centralizada,Suministros,Selección Abreviada de Menor Cuantía,V1.50131612,SUMINISTRO DE HUEVOS CON DESTINO A LOS COMEDORES DE TROPA DE LA BR-4 Y BR-14; ADMINISTRADOS POR LA AGENCIA LOGÍSTICA DE LAS FUERZAS MILITARES REGIONAL ANTIOQUIA CHOCO Y OTRAS POSIBLES UNIDADES QUE LO REQUIERAN,2019-06-23,2019-06-13,2019-11-29,Dia(s),Transporte incluido,Distribuidora Antioqueña AM,21788564,Si,No,Mujer,CO,390000000.0,389999685,389999685,315,0,No,No,Distribuido,Funcionamiento,Cerrado,Si,No,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.841446&isFromPublicArea=True&isModal=true&asPopupView=true'}
2,CO1.PCCNTR.1000901,FUERZA AEROESPACIAL COLOMBIANA,Distrito Capital de Bogotá,Bogotá,Nacional,defensa,Ejecutivo,Centralizada,Compraventa,Mínima cuantía,V1.53102701,ADQUISICIÓN DE PRESILLAS BORDADAS AZULES EN CANUTILLO PARA OFICIALES DE INSIGNIA DE LA FUERZA AÉREA COLOMBIANA CONFORME ANEXO TECNICO,2019-06-18,2019-06-24,2019-09-30,No definido,DAP - Entregado en un punto (lugar de destino convenido),FANNY JANETH GUTIERREZ PRIETO,20484821,No,No,Mujer,CO,11970000.0,11970000,11970000,0,0,No Definido,Si,Distribuido,Funcionamiento,terminado,No,No,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.846657&isFromPublicArea=True&isModal=true&asPopupView=true'}
3,CO1.PCCNTR.1000908,AGENCIA LOGISTICA DE LAS FUERZAS MILITARES,Distrito Capital de Bogotá,Bogotá,Nacional,defensa,Ejecutivo,Centralizada,Compraventa,Licitación pública,V1.41115300,ADQUISICIÓN DE UN SISTEMA DE SONAR 3D DE ALTA RESOLUCIÓN CON DESTINO AL DEPARTAMENTO DE BUCEO Y SALVAMENTO,2019-07-25,2019-07-12,2019-11-30,Dia(s),Como acordado previamente,CASCO ANTIGUO,76044753,No,No,Otro,CO,733992000.0,0,0,733992000,0,No,No,Distribuido,Funcionamiento,Cerrado,Si,No,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.826239&isFromPublicArea=True&isModal=true&asPopupView=true'}
4,CO1.PCCNTR.1000911,PARQUES NACIONALES NATURALES DE COLOMBIA - DIRECCION TERRITORIAL CARIBE,Bolívar,Cartagena,Territorial,Ambiente y Desarrollo Sostenible,Ejecutivo,Centralizada,Suministros,Mínima cuantía,V1.92121701,Suministro de equipos para la instalación de todas las cámaras necesarias para el monitoreo; para ser instalados en\nlas sedes oper

In [8]:
secop_df.tail

<bound method NDFrame.tail of                id_contrato  \
0       CO1.PCCNTR.1000001   
1       CO1.PCCNTR.1000507   
2       CO1.PCCNTR.1000901   
3       CO1.PCCNTR.1000908   
4       CO1.PCCNTR.1000911   
...                    ...   
196386   CO1.PCCNTR.999805   
196387   CO1.PCCNTR.999806   
196388   CO1.PCCNTR.999807   
196389   CO1.PCCNTR.999810   
196390   CO1.PCCNTR.999902   

                                                                    nombre_entidad  \
0       PARQUES NACIONALES NATURALES DE COLOMBIA - DIRECCION  TERRITORIAL AMAZONIA   
1                                       AGENCIA LOGISTICA DE LAS FUERZAS MILITARES   
2                                                   FUERZA AEROESPACIAL COLOMBIANA   
3                                       AGENCIA LOGISTICA DE LAS FUERZAS MILITARES   
4          PARQUES NACIONALES NATURALES DE COLOMBIA - DIRECCION TERRITORIAL CARIBE   
...                                                                            ...   
196386 

In [9]:
# 1. ¿Cuántos IDs únicos hay vs. cuántas filas totales?
print("Total de filas:", len(secop_df))
print("IDs únicos:", secop_df['id_contrato'].nunique())

# 2. Si los números no coinciden, hay duplicados
duplicados = secop_df['id_contrato'].duplicated().sum()
print("Filas con id_contrato repetido:", duplicados)

Total de filas: 196391


IDs únicos: 196391
Filas con id_contrato repetido: 0


In [10]:
secop_df.sample(10)

,id_contrato,nombre_entidad,departamento,ciudad,orden,sector,rama,entidad_centralizada,tipo_de_contrato,modalidad_de_contratacion,codigo_de_categoria_principal,objeto_del_contrato,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,duracion_del_contrato,condiciones_de_entrega,proveedor_adjudicado,documento_proveedor,es_pyme,es_grupo,genero_representante_legal,nacionalidad_representante_legal,valor_del_contrato,valor_pagado,valor_facturado,valor_pendiente_de_pago,dias_adicionados,habilita_pago_adelantado,el_contrato_puede_ser_prorrogado,origen_de_los_recursos,destino_gasto,estado_contrato,liquidacion,obligacion_ambiental,urlproceso
162491,CO1.PCCNTR.7938609,MUNICIPIO DE FRONTINO,Antioquia,Frontino,Territorial,"Vivienda, Ciudad y Territorio",Ejecutivo,Centralizada,Suministros,Mínima cuantía,V1.44103103,SUMINISTRO DE PAPELERÍA PARA EL FUNCIONAMIENTO DE LAS DEPENDENCIAS DE LA ADMINISTRACIÓN MUNICIPAL DE FRONTINO - ANTIOQUIA,2025-05-31,2025-05-31,2025-06-10,10 Dia(s),NXTWY.DLVY.6,PAPELERIA EL CID S.A.S.,800021033,Si,No,Hombre,CO,3.306364e+07,0,0,33063641,0,No,No,Distribuido,Funcionamiento,En ejecución,No,No,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8171059&isFromPublicArea=True&isModal=true&asPopupView=true'}
148053,CO1.PCCNTR.7158830,MUNICIPIO DE FREDONIA,Antioquia,Fredonia,Territorial,Servicio Público,Ejecutivo,Centralizada,Suministros,Mínima cuantía,V1.90101500,PRESTACIÓN DE SERVICIOS DE ALIMENTACIÓN PARA LOS MIEMBROS DE LA FUERZA PÚBLICA QUE APOYAN LOS PROGRAMAS DE SEGURIDAD DE FORMA ITINERANTE EN EL MUNICIPIO DE FREDONIA,2024-12-20,2024-12-20,2024-12-31,12 Dia(s),NXTWY.DLVY.2,ALEJANDRA MARIA ARANGO ACEVEDO,43414622,No,No,Mujer,CO,3.550000e+07,0,0,35500000,0,No,No,Distribuido,Inversión,En ejecución,No,No,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7211174&isFromPublicArea=True&isModal=true&asPopupView=true'}
96973,CO1.PCCNTR.5246457,Gobernación Norte de Santander*,Norte de Santander,Cúcuta,Territorial,No aplica/No pertenece,Ejecutivo,Centralizada,Suministros,Selección abreviada subasta inversa,V1.53101500,EL SUMINISTRO DE TRES (3) BONOS CONVERTIBLES EN TRES (3) DOTACIONES DE CALZADO Y VESTIDO DE LABOR CORRESPONDIENTES A LA VIGENCIA 2018; PARA EL PERSONAL DOCENTE Y DIRECTIVO DOCENTE QUE PRESTA SUS SERVICIOS EN LOS 39 MUNICIPIOS NO CERTIFICADOS DEL DEPARTAMENTO NORTE DE SANTANDER A TRAVÉS DE LA MODALIDAD DE BONOS U ÓRDENES DE COMPRA EN LOS MUNICIPIOS DE OCAÑA; PAMPLONA Y CÚCUTA.,2023-07-28,2023-08-15,2023-12-30,5 Mes(es),Como acordado previamente,CRISALLTEX S.A.S,816007113,Si,No,Mujer,CO,1.483200e+09,294679201,540165602,1188520798,0,No,No,Distribuido,Inversión,terminado,Si,No,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4720017&isFromPublicArea=True&isModal=true&asPopupView=true'}
87192,CO1.PCCNTR.4858747,HOSMIL,Distrito Capital de Bogotá,No Definido,Nacional,defensa,Ejecutivo,Descentralizada,Compraventa,Mínima cuantía,V1.23121600,ADQUISICIÓN; PREINSTALACIÓN; INSTALACIÓN Y PUESTA EN FUNCIONAMIENTO DE UNA MAQUINA BORDADORA INDUSTRIAL DE ÚLTIMA GENERACIÓN PARA EL TALLER DE CONFECCIONES A CARGO DEL SERVICIO DE LAVANDERÍA DEL HOSPITAL MILITAR CENTRAL,2023-04-14,2023-04-19,2023-06-30,3 Mes(es),Como acordado previamente,GROUP MLS SAS,900068178,Si,No,Mujer,CO,2.499000e+07,24990000,24990000,0,0,No,No,Distribuido,Funcionamiento,Cerrado,Si,No,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4251563&isFromPublicArea=True&isModal=true&asPopupView=true'}
4728,CO1.PCCNTR.1127118,INVIAS,Distrito Capital de Bogotá,Bogotá,Nacional,Transporte,Ejecutivo,Centralizada,Suministros,Mínima cuantía,V1.72141000,SUMINISTRO DE MATERIAL DE AFIRMADO PARA BACHEO EN LA CARRETERA 6501; VILLAGARZON - SAN JOSE DEL FRAGUA; SECTOR PR 24+0500 (RIO CAQUETA) - PR 55+0600(PUERTO BELLO); CON MANO DE OBRA DE LAS COOPERATIVAS DE TRABAJO ASOCIA

In [11]:
secop_df.describe()

,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,valor_del_contrato,valor_pagado,valor_facturado,valor_pendiente_de_pago,dias_adicionados
count,196391,193951,196391,1.963910e+05,1.963910e+05,1.963910e+05,1.963910e+05,196391.000000
mean,2023-02-27 22:45:53.539622,2023-03-01 13:02:29.326376,2023-06-11 04:37:50.596921,6.582426e+10,1.194911e+08,1.484092e+08,6.570476e+10,5.107485
min,2019-01-01 00:00:00,2016-06-27 00:00:00,0202-12-31 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,-8.418488e+07,0.000000
25%,2021-09-10 00:00:00,2021-09-17 00:00:00,2021-12-20 00:00:00,1.202121e+07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
50%,2023-05-30 00:00:00,2023-06-05 00:00:00,2023-10-27 00:00:00,3.360000e+07,0.000000e+00,3.207328e+06,6.717571e+06,0.000000
75%,2024-11-07 00:00:00,2024-11-14 00:00:00,2024-12-31 00:00:00,1.000000e+08,2.812077e+07,3.708622e+07,4.241189e+07,0.000000
max,2025-12-31 00:00:00,2026-12-16 00:00:00,2065-01-31 00:00:00,1.285845e+16,5.944034e+11,5.945032e+11,1.285845e+16,14276.000000
std,NaN,NaN,NaN,2.901535e+13,2.444116e+09,2.969810e+09,2.901535e+13,40.662426


In [12]:
secop_df['sector'].nunique()

26

In [13]:
secop_df['sector'].value_counts()

sector
Servicio Público                                      37390
defensa                                               34585
No aplica/No pertenece                                24465
Salud y Protección Social                             22037
Ley de Justicia                                       20377
Trabajo                                               14759
Educación Nacional                                    10231
Ambiente y Desarrollo Sostenible                       8109
Transporte                                             4655
Cultura                                                2792
deportes                                               2610
Hacienda y Crédito Público                             2441
Industria                                              1807
Vivienda, Ciudad y Territorio                          1497
Planeación                                             1319
agricultura                                            1254
interior                         

In [14]:
secop_df['tipo_de_contrato'].nunique()

2

In [15]:
secop_df['tipo_de_contrato'].value_counts()

tipo_de_contrato
Suministros    106632
Compraventa     89759
Name: count, dtype: int64

In [16]:
secop_df['fecha_de_firma'].dt.year.value_counts().sort_index()

fecha_de_firma
2019    16425
2020    20135
2021    24043
2022    28070
2023    32971
2024    34634
2025    40113
Name: count, dtype: int64

In [17]:
secop_df['valor_del_contrato'].hist(bins=50)

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

In [ ]:
secop_df['valor_del_contrato'].quantile([0.01, 0.25, 0.5, 0.75, 0.99])

0.01    7.010680e+05
0.25    1.202121e+07
0.50    3.360000e+07
0.75    1.000000e+08
0.99    4.118482e+09
Name: valor_del_contrato, dtype: float64

In [18]:
secop_df['modalidad_de_contratacion'].value_counts()

modalidad_de_contratacion
Mínima cuantía                                                 117753
Selección abreviada subasta inversa                             32907
Contratación régimen especial                                   19250
Selección Abreviada de Menor Cuantía                             8822
Contratación régimen especial (con ofertas)                      6154
Contratación Directa (con ofertas)                               5001
Contratación directa                                             3938
Licitación pública                                               2330
Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes       236
Name: count, dtype: int64

In [19]:
secop_df['orden'].value_counts()

orden
Territorial             97293
Nacional                96680
Corporación Autónoma     2409
No Definido                 9
Name: count, dtype: int64

In [20]:
secop_df['destino_gasto'].value_counts()

destino_gasto
Funcionamiento    117097
Inversión          78383
No aplica            532
No Definido          379
Name: count, dtype: int64

## Hallazgos

**Pendientes de limpieza detectados y su tratamiento:**

- Nombres de columna con problemas de encoding (`duraci_n_del_contrato`, `g_nero_representante_legal`, `liquidaci_n`, `obligaci_n_ambiental`) — **corregidos** renombrando a `duracion_del_contrato`, `genero_representante_legal`, `liquidacion` y `obligacion_ambiental`, siguiendo la misma convención sin tildes que ya usan las demás columnas.
- La granularidad es una fila = un `id_contrato` único (196,391 filas, sin duplicados).
- Nulos verdaderos: `fecha_de_inicio_del_contrato` (2,440 filas, 1.2%) y `documento_proveedor` (4 filas) — se dejan como `NaN`, sin imputar.
- Fechas fuera de rango o inconsistentes: `fecha_de_inicio_del_contrato` (378 filas fuera de 2016–2025, máx. `2026-12-16`), `fecha_de_fin_del_contrato` (5,605 filas fuera de rango, mín. `0202-12-31`, máx. `2065-01-31`), y 125 filas con `fecha_de_fin < fecha_de_inicio`. **Tratamiento:** no se eliminan filas ni se imputan fechas; se marcan con la columna booleana `fecha_sospechosa` (2.9% del total, 5,730 filas) para poder excluirlas puntualmente de análisis de duración/plazo más adelante sin perder el resto de la información del contrato.
- `valor_pendiente_de_pago` negativo (144 filas): se verificó que el 100% de esos casos tiene `valor_pagado > valor_del_contrato` — es un patrón de sobrepago real, no un error de captura. **Tratamiento:** se deja el valor sin modificar; queda como posible señal de interés para la focalización de supervisión.
- Categorías centinela (`No Definido` / `No aplica`) usadas como texto en vez de nulo real, con tasas de faltantes altas en varias columnas clave: `condiciones_de_entrega` (25.4%), `ciudad` (21.2%), `genero_representante_legal` (18.2%), `habilita_pago_adelantado` (14.5%), `duracion_del_contrato` (4.8%), `departamento` (3.8%), y en menor medida `orden`, `sector`, `rama`, `destino_gasto`, `objeto_del_contrato`, `proveedor_adjudicado`, `documento_proveedor` y `nacionalidad_representante_legal`. **Tratamiento:** estandarizadas a `NA` explícito en todas las columnas de texto, para que dejen de contarse como una categoría de negocio más en los `value_counts` del análisis univariado.